# Chapter 5 gallery — MM regression

Reproduces `algae.R` (Ex 5.4), `ExactFit.R` (Ex 5.5), `wood.R` (Ex 5.2) and `step.R` (Ex 5.3) via `lmrobdet_mm` + `step_lmrobdet`.

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## algae — MM regression with a dot formula (Example 5.4)

`algae.R` fits `lmrobdetMM(V12 ~ .)` on the algae-bloom data (90×12) and contrasts it with LS. We reproduce the robust fit and its standardized-residual diagnostic (Fig 5.15).

In [ ]:
algae = rpm.datasets.algae()
ctrl = rpm.lmrobdet_control(bb=0.5, efficiency=0.85, family='bisquare')
rob = rpm.lmrobdet_mm('V12 ~ .', algae, control=ctrl)
print('robust scale:', round(float(rob.scale), 4))
print('n coefficients:', len(rob.coefficients))
resid = np.asarray(rob.residuals, dtype=float)
std_resid = resid / float(rob.scale)
outliers = np.flatnonzero(np.abs(std_resid) > 2.5)
print('rows with |std resid| > 2.5:', (outliers + 1).tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(np.arange(1, len(std_resid)+1), std_resid, c='k', s=18)
for h in (-2.5, 0, 2.5):
    ax.axhline(h, ls='--', c='gray')
ax.set_xlabel('index'); ax.set_ylabel('standardized robust residual')
ax.set_title('algae — Fig 5.15 analogue')
fig.savefig(FIG_DIR / 'ch5_algae_resid.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('robust residuals expose the outliers LS hides')

### Strict-tier cross-check vs direct R `lmrobdetMM`

In [ ]:
# lmrobdetMM is deterministic (Peña–Yohai initial), so no seeding is needed.
rob_chk = rpm.lmrobdet_mm('V12 ~ .', algae, control=ctrl)
ro.r('data(algae); cont <- lmrobdet.control(bb=0.5, efficiency=0.85, family="bisquare")')
ro.r('ra <- lmrobdetMM(V12 ~ ., data=algae, control=cont)')
r_coef = np.asarray(ro.r('as.numeric(ra$coefficients)'), dtype=float)
print('coefficients bit-equal to R:', np.array_equal(rob_chk.coefficients, r_coef))

## ExactFit — MM vs LS on a 1/3-contaminated line (Example 5.5)

`ExactFit.R` builds 100 'good' points on `y = x` plus 50 outliers on `y = -2x`, then fits LS and MM. The MM line locks onto the majority; LS is pulled toward the contamination.

In [ ]:
# Reproduce the data generation exactly via R's RNG, then fit in Python.
ro.r('''set.seed(1003); n<-100; m<-50; rr<-rnorm(m)
x1<-sort(rnorm(n)); x2<-sort(rr)*2; sig<-0.1
y1<-x1+sig*rnorm(n); y2<- -x2+sig*rnorm(m)
xe<-c(x1,x2); ye<-c(y1,y2)''')
xe = np.asarray(ro.r('xe'), dtype=float); ye = np.asarray(ro.r('ye'), dtype=float)
mm = rpm.lmrobdet_mm('y ~ x', pd.DataFrame({'x': xe, 'y': ye}))
Xe = np.c_[np.ones(len(xe)), xe]
ls = np.linalg.lstsq(Xe, ye, rcond=None)[0]
print('LS slope :', round(ls[1], 3), '(pulled toward the outliers)')
print('MM slope :', round(float(mm.coefficients[1]), 3), '(tracks the good majority, ~1)')

In [ ]:
grid = np.linspace(xe.min(), xe.max(), 50)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(xe, ye, c='gray', s=14)
ax.plot(grid, ls[0] + ls[1]*grid, 'b-', lw=2, label='LS')
ax.plot(grid, mm.coefficients[0] + mm.coefficients[1]*grid, 'r-', lw=2, label='MM')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); ax.set_title('ExactFit — Fig 5.16 analogue')
fig.savefig(FIG_DIR / 'ch5_exactfit.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('done')

## wood — MM regression on the robustbase wood data (Example 5.2)

`wood.R` loads `wood` from **robustbase** (cross-package) and fits `lmrobdetMM(y ~ .)`. We use `rpm.datasets.load('robustbase', 'wood')`.

In [ ]:
wood = rpm.datasets.load('robustbase', 'wood')
print('columns:', list(wood.columns))
ctrl = rpm.lmrobdet_control(bb=0.5, efficiency=0.85, family='bisquare')
wMM = rpm.lmrobdet_mm('y ~ .', wood, control=ctrl)
wresid = np.abs(np.asarray(wMM.residuals, dtype=float))
flagged = np.flatnonzero(wresid > 2.5 * float(wMM.scale))
print('robust scale:', round(float(wMM.scale), 5))
print('outlying rows (|resid| > 2.5*scale):', (flagged + 1).tolist())

## step — robust stepwise model selection (Example 5.3)

`step.R` builds a 6-predictor design with planted outliers, fits the full `lmrobdetMM`, then runs `step.lmrobdetMM` (robust backward selection by RFPE). The true model uses only the first three predictors.

In [ ]:
import pandas as pd
# Reproduce step.R's data exactly via R's RNG.
ro.r('''set.seed(300); X<-matrix(rnorm(50*6),50,6); beta<-c(1,1,1,0,0,0)
y<-as.vector(X%*%beta)+1+rnorm(50); y[1:6]<-seq(30,55,5)
for (i in 1:6) X[i,]<-c(X[i,1:3],i/2,i/2,i/2); Z<-as.data.frame(cbind(y,X))''')
Z = pd.DataFrame(np.asarray(ro.r('as.matrix(Z)'), dtype=float),
                 columns=['y','V2','V3','V4','V5','V6','V7'])
ctrl = rpm.lmrobdet_control(bb=0.5, efficiency=0.85, family='bisquare')
obj = rpm.lmrobdet_mm('y ~ .', Z, control=ctrl)
sel = rpm.step_lmrobdet(obj)
print('final model:', sel.final_formula)
print('final coefficients:', dict(zip(sel.coef_names, np.round(sel.coefficients, 3))))